# R08-H57 - One estimator family, three inventories (Good-Turing completeness)

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-07 <br>
**Pipeline stage**: R08 conformist round, coverage bounds <br>
**Graph**: rebuilt CPAP graph (neo4j2, read-only) <br>

Applies the promoted missing-mass UCB (H32) to two more categorical inventories: the relationship-type
inventory and per-entity-type property-KEY inventories. Missing mass m = n1/N (types/keys seen once over
total observations); UCB = n1/N + z*sqrt(n1+1)/N per the H32 convention (z=1.96). Coverage bound = 1 - UCB.
The refusal-correlation clause needs probe cycles - bounds reported now, correlation marked PENDING.

In [1]:
import json, datetime, collections, math
from neo4j import GraphDatabase
from rich import print as rprint
NEO4J_URI='bolt://user-konrad.jelen-kgf-neo4j2:7687'  # read-only neo4j2 (NOT .env / live graph)
Z=1.96
def gt(counts):
    # counts: dict category -> observation count
    N=sum(counts.values());
    if N==0: return dict(N=0,n1=0,missing_mass=None,ucb=None,coverage_lb=None,n_categories=0)
    n1=sum(1 for c in counts.values() if c==1)
    mm=n1/N; ucb=mm + Z*math.sqrt(n1+1)/N
    return dict(N=N,n1=n1,n_categories=len(counts),missing_mass=mm,ucb=min(ucb,1.0),coverage_lb=max(0.0,1-ucb))

## Inventory 1 - relationship types

In [2]:
drv=GraphDatabase.driver(NEO4J_URI, auth=('neo4j','kgfoundry'), notifications_min_severity='OFF')
with drv.session() as s:
    reltypes=[r['relationshipType'] for r in s.run('CALL db.relationshipTypes()')]
    relcounts={}
    for rt in reltypes:
        relcounts[rt]=s.run(f'MATCH ()-[r:`{rt}`]->() RETURN count(r) AS c').single()['c']
SYSTEM={'ABOUT','MENTIONED_IN','SAME_AS','SIMILAR_TO'}  # provenance / resolution edges, not domain semantics
semantic={k:v for k,v in relcounts.items() if k not in SYSTEM}
all_gt=gt(relcounts); sem_gt=gt(semantic)
rprint('all relationship types:', all_gt)
rprint('semantic-only (excl ABOUT/MENTIONED_IN/SAME_AS/SIMILAR_TO):', sem_gt)
singletons=[k for k,v in semantic.items() if v==1]
rprint(f'semantic singleton rel types (n1): {len(singletons)} ->', singletons)

all relationship types:
{
    'N': 31593,
    'n1': 17,
    'n_categories': 71,
    'missing_mass': 0.0005380938815560409,
    'ucb': 0.0008013033186703951,
    'coverage_lb': 0.9991986966813297
}

semantic-only (excl ABOUT/MENTIONED_IN/SAME_AS/SIMILAR_TO):
{
    'N': 4106,
    'n1': 17,
    'n_categories': 67,
    'missing_mass': 0.004140282513395032,
    'ucb': 0.0061655079753418895,
    'coverage_lb': 0.9938344920246581
}

semantic singleton rel types (n1): 17 ->
[
    'ILLUMINATES',
    'REQUIRED_BEFORE',
    'LOCATED_IN',
    'USES_SOFTWARE',
    'CONNECTS_TO',
    'FACILITATES',
    'DETERMINES',
    'AFFECTS',
    'RECOMMENDED_FOR',
    'MAXIMIZES_LIFESPAN_OF',
    'CAUSES_DAMAGE_TO',
    'OCCURS_IN',
    'COMPONENT_OF',
    'TREATED_BY',
    'USES_FEATURE',
    'MEASURES',
    'BENEFIT_OF'
]

## Inventory 2 - per-entity-type property keys

In [3]:
with drv.session() as s:
    labels=[l for l in [r['label'] for r in s.run('CALL db.labels()')]
            if l not in ('KGFControl','KGFLock','KGFDocument','Chunk','Proposition','KGFCommunity','KGFEntityVersion')]
    per_type={}
    for lb in labels:
        rows=s.run(f'MATCH (e:`{lb}`) RETURN e AS e').data()
        if not rows: continue
        keycount=collections.Counter()
        for r in rows:
            for k,v in r['e'].items():
                if k.startswith('prop_') and v is not None and v!='':
                    keycount[k]+=1
        g=gt(keycount); g['n_entities']=len(rows)
        per_type[lb]=g
drv.close()

rows_sorted=sorted([(lb,g) for lb,g in per_type.items() if g['N']>0], key=lambda x:-(x[1]['ucb'] or 0))
rprint(f'{"type":22s} {"ents":>5} {"keys":>5} {"N":>5} {"n1":>4} {"mm":>7} {"UCB":>7} {"cov_lb":>7}')
for lb,g in rows_sorted:
    rprint(f'{lb:22s} {g["n_entities"]:>5} {g["n_categories"]:>5} {g["N"]:>5} {g["n1"]:>4} {g["missing_mass"]:>7.3f} {g["ucb"]:>7.3f} {g["coverage_lb"]:>7.3f}')
high_ucb=[lb for lb,g in per_type.items() if (g['ucb'] or 0)>0.2 and g['N']>0]
rprint(f'[bold]cohorts with UCB > 0.2 (predicted refusal-heavy): {len(high_ucb)}[/bold] ->', high_ucb)

type                    ents  keys     N   n1      mm     UCB  cov_lb

Condition                 39     1     1    1   1.000   1.000   0.000

Standard                  58    48    54   44   0.815   1.000   0.000

TestProtocol              31    35    42   31   0.738   1.000   0.000

Person                    13     6     8    4   0.500   1.000   0.000

...

ConnectivityDevice        60    28    60   13   0.217   0.339   0.661

EventType                139    60   165   32   0.194   0.262   0.738

Entity                  2798  2050  5982 1451   0.243   0.255   0.745

Accessory                999   519  2548  315   0.124   0.137   0.863

cohorts with UCB > 0.2 (predicted refusal-heavy): 20 ->
[
    'Entity',
    'CPAPDevice',
    'OperatingMode',
    'ComfortFeature',
    'Feature',
    'ClinicalFeature',
    'Software',
    'Manufacturer',
    'DataStorageDevice',
    'ConnectivityDevice',
    'EventType',
    'ProductModel',
    'Specification',
    'Organization',
    'MedicalCondition',
    'Condition',
    'Standard',
    'TestProtocol',
    'Person',
    'Component'
]

## Verdict + report

In [4]:
# registered prediction: relationship-type missing mass <=5%; per-type property coverage varies widely
rel_mm=sem_gt['missing_mass']; rel_ucb=sem_gt['ucb']
covs=[g['coverage_lb'] for g in per_type.values() if g['N']>0]
spread=max(covs)-min(covs) if covs else 0
report=dict(hypothesis='R08-H57',
  relationship_inventory_all=all_gt, relationship_inventory_semantic=sem_gt,
  semantic_singleton_types=singletons,
  property_key_inventories={lb:g for lb,g in per_type.items()},
  high_ucb_cohorts=high_ucb, coverage_lb_spread=spread,
  rel_missing_mass=rel_mm, rel_missing_mass_bar='<=0.05 predicted',
  refusal_correlation='PENDING - requires per-wave probe cycles (registration: report bounds now, correlation later)',
  bar='coverage bounds rank-correlate with refusals; refuted if flat across cohorts',
  verdict='BOUNDS REPORTED - correlation clause PENDING probe cycles',
  note=f'rel-type missing mass {rel_mm:.3f} (bar <=0.05: {"holds" if rel_mm<=0.05 else "exceeds"}); property coverage_lb spread {spread:.2f} across {len(covs)} cohorts (widely-varying: prediction holds)')
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
path=f'../reports/good-turing-inventories-h57-{stamp}.json'
json.dump(report,open(path,'w'),indent=2)
rprint(f'rel-type missing mass {rel_mm:.3f} (UCB {rel_ucb:.3f}); property coverage_lb spread {spread:.2f}')
rprint('[bold]bounds reported; refusal-correlation clause PENDING probe cycles[/bold]')
rprint('wrote',path)

rel-type missing mass 0.004 (UCB 0.006); property coverage_lb spread 0.86

bounds reported; refusal-correlation clause PENDING probe cycles

wrote ../reports/good-turing-inventories-h57-20260707-092806.json